In [2]:
!pip install xlrd

import pandas as pd

file_path = r"C:\DI\[노드7] Vibe하게 데이터 분석하기(D035~D039)\[과제] 대시보드 제작\sample_-_superstore.xls"  # 본인 경로로 수정

orders = pd.read_excel(file_path, sheet_name="Orders")
people = pd.read_excel(file_path, sheet_name="People")
returns = pd.read_excel(file_path, sheet_name="Returns")

# 1. 기본 구조
print("=== Orders 구조 ===")
print(orders.shape)
print(orders.dtypes)
print()

# 2. 결측치
print("=== 결측치 개수 ===")
print(orders.isnull().sum())
print()

# 3. 중복 확인
print("=== 중복 확인 ===")
print("Row ID 중복 개수:", orders["Row ID"].duplicated().sum())
print("완전 중복 행 개수:", orders.duplicated().sum())
print("Order ID+Product ID 조합 중복 개수:",
      orders.duplicated(subset=["Order ID", "Product ID"]).sum())

=== Orders 구조 ===
(10194, 21)
Row ID                     int64
Order ID                     str
Order Date        datetime64[us]
Ship Date         datetime64[us]
Ship Mode                    str
Customer ID                  str
Customer Name                str
Segment                      str
Country/Region               str
City                         str
State/Province               str
Postal Code               object
Region                       str
Product ID                   str
Category                     str
Sub-Category                 str
Product Name                 str
Sales                    float64
Quantity                   int64
Discount                 float64
Profit                   float64
dtype: object

=== 결측치 개수 ===
Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Customer Name     0
Segment           0
Country/Region    0
City              0
State/Province    0
Postal Code       0
Region 

In [3]:
# Order ID + Product ID 조합 중복 건 상세 확인
dup_mask = orders.duplicated(subset=["Order ID", "Product ID"], keep=False)
dup_rows = orders[dup_mask].sort_values(["Order ID", "Product ID"])

print("중복 관련 총 행 수:", len(dup_rows))
print(dup_rows[["Row ID", "Order ID", "Product ID", "Order Date",
                "Quantity", "Discount", "Sales", "Profit"]].to_string())

# 

중복 관련 총 행 수: 22
      Row ID        Order ID       Product ID Order Date  Quantity  Discount     Sales    Profit
2017    2018  CA-2023-123625  FUR-FU-10004093 2023-12-29         8       0.0   101.120   37.4144
2019    2020  CA-2023-123625  FUR-FU-10004093 2023-12-29         2       0.0    68.460   20.5380
2020    2021  CA-2023-123625  OFF-FA-10000089 2023-12-29         2       0.0    68.460   20.5380
2022    2023  CA-2023-123625  OFF-FA-10000089 2023-12-29         5       0.0    19.600    9.6040
1698    1699  CA-2023-153623  FUR-FU-10002501 2023-11-24         8       0.0    99.120   35.4144
1699    1700  CA-2023-153623  FUR-FU-10002501 2023-11-24         8       0.0    99.120   35.4144
390      391  US-2023-150119  FUR-CH-10002965 2023-04-23         2       0.3   281.372  -12.0588
391      392  US-2023-150119  FUR-CH-10002965 2023-04-23         2       0.3   281.372  -12.0588
2879    2880  US-2024-103135  OFF-BI-10000069 2024-07-24         9       0.0   135.090   62.1414
2880    2881  

In [4]:
# Row ID를 제외한 나머지 컬럼 전부를 기준으로 완전 중복 판단
subset_cols = [col for col in orders.columns if col != "Row ID"]

# 제거 전, 어떤 행이 중복으로 잡히는지 재확인
exact_dup_mask = orders.duplicated(subset=subset_cols, keep=False)
print("완전 중복으로 잡힌 행 수:", exact_dup_mask.sum())
print(orders.loc[exact_dup_mask, ["Row ID", "Order ID", "Product ID", "Quantity", "Discount", "Sales", "Profit"]])
print()

# 중복 중 첫 번째 행만 남기고 제거
orders_clean = orders.drop_duplicates(subset=subset_cols, keep="first").reset_index(drop=True)

print("제거 전 행 수:", len(orders))
print("제거 후 행 수:", len(orders_clean))
print("제거된 행 수:", len(orders) - len(orders_clean))

완전 중복으로 잡힌 행 수: 4
      Row ID        Order ID       Product ID  Quantity  Discount    Sales  \
390      391  US-2023-150119  FUR-CH-10002965         2       0.3  281.372   
391      392  US-2023-150119  FUR-CH-10002965         2       0.3  281.372   
1698    1699  CA-2023-153623  FUR-FU-10002501         8       0.0   99.120   
1699    1700  CA-2023-153623  FUR-FU-10002501         8       0.0   99.120   

       Profit  
390  -12.0588  
391  -12.0588  
1698  35.4144  
1699  35.4144  

제거 전 행 수: 10194
제거 후 행 수: 10192
제거된 행 수: 2


In [5]:
# 날짜 논리 확인: Ship Date가 Order Date보다 빠른 경우
invalid_dates = orders_clean[orders_clean["Ship Date"] < orders_clean["Order Date"]]
print("Ship Date < Order Date인 행 수:", len(invalid_dates))
print(invalid_dates[["Row ID", "Order ID", "Order Date", "Ship Date"]].to_string())

# 배송 소요일 분포
lead_time = (orders_clean["Ship Date"] - orders_clean["Order Date"]).dt.days
print()
print("배송 소요일(days) 기초통계:")
print(lead_time.describe())

Ship Date < Order Date인 행 수: 0
Empty DataFrame
Columns: [Row ID, Order ID, Order Date, Ship Date]
Index: []

배송 소요일(days) 기초통계:
count    10192.000000
mean         3.960655
std          1.741605
min          0.000000
25%          3.000000
50%          4.000000
75%          5.000000
max         11.000000
dtype: float64


In [6]:
# 5. 수치형 값 범위 확인

# Quantity, Sales가 0 이하인 경우 (있으면 안 되는 값)
invalid_qty = orders_clean[orders_clean["Quantity"] <= 0]
invalid_sales = orders_clean[orders_clean["Sales"] <= 0]
print("Quantity <= 0인 행 수:", len(invalid_qty))
print("Sales <= 0인 행 수:", len(invalid_sales))
print()

# Discount가 0~1 범위를 벗어나는 경우
invalid_discount = orders_clean[(orders_clean["Discount"] < 0) | (orders_clean["Discount"] > 1)]
print("Discount가 0~1 범위 밖인 행 수:", len(invalid_discount))
print()

# 전체 수치형 컬럼 기초 통계로 극단값 감 잡기
print(orders_clean[["Sales", "Quantity", "Discount", "Profit"]].describe())
print()

# Profit 하위 1%, 상위 1% 값 확인 (극단적 손실/이익 케이스)
print("Profit 하위 1%:", orders_clean["Profit"].quantile(0.01))
print("Profit 상위 1%:", orders_clean["Profit"].quantile(0.99))

Quantity <= 0인 행 수: 0
Sales <= 0인 행 수: 0

Discount가 0~1 범위 밖인 행 수: 0

              Sales      Quantity      Discount        Profit
count  10192.000000  10192.000000  10192.000000  10192.000000
mean     228.233307      3.791601      0.155386     28.676752
std      619.966122      2.228075      0.206258    232.487565
min        0.444000      1.000000      0.000000  -6599.978000
25%       17.220000      2.000000      0.000000      1.760800
50%       53.890000      3.000000      0.200000      8.690000
75%      209.500000      5.000000      0.200000     29.289775
max    22638.480000     14.000000      0.800000   8399.976000

Profit 하위 1%: -312.51962599999996
Profit 상위 1%: 570.9399010000017


In [7]:
# Profit 극단값 상위/하위 5건을 Sales, Quantity, Discount와 함께 확인
top5_profit = orders_clean.nlargest(5, "Profit")
bottom5_profit = orders_clean.nsmallest(5, "Profit")

cols = ["Row ID", "Order ID", "Product ID", "Category", "Sub-Category",
        "Sales", "Quantity", "Discount", "Profit"]

print("=== Profit 상위 5건 ===")
print(top5_profit[cols].to_string())
print()
print("=== Profit 하위 5건 (손실 큰 순) ===")
print(bottom5_profit[cols].to_string())

=== Profit 상위 5건 ===
      Row ID        Order ID       Product ID         Category Sub-Category     Sales  Quantity  Discount     Profit
5892    5895  US-2025-118689  TEC-CO-10004722       Technology      Copiers  17499.95         5       0.0  8399.9760
7253    7256  US-2026-140151  TEC-CO-10004722       Technology      Copiers  13999.96         4       0.0  6719.9808
9515    9518  US-2026-166709  TEC-CO-10004722       Technology      Copiers  10499.97         3       0.0  5039.9856
6672    6675  US-2025-117121  OFF-BI-10000545  Office Supplies      Binders   9892.74        13       0.0  4946.3700
1202    1204  US-2023-116904  OFF-BI-10001120  Office Supplies      Binders   9449.95         5       0.0  4630.4755

=== Profit 하위 5건 (손실 큰 순) ===
      Row ID        Order ID       Product ID         Category Sub-Category     Sales  Quantity  Discount     Profit
6396    6399  US-2025-108196  TEC-MA-10000418       Technology     Machines  4499.985         5       0.7 -6599.9780
9316    9319

In [8]:
# 6. 범주형 값 일관성 확인
categorical_cols = ["Ship Mode", "Segment", "Country/Region", "Region",
                     "Category", "Sub-Category"]

for col in categorical_cols:
    print(f"=== {col} (고유값 {orders_clean[col].nunique()}개) ===")
    print(orders_clean[col].value_counts())
    print()

# 문자열 앞뒤 공백/대소문자 이슈 확인 (strip 전후 값 비교)
for col in categorical_cols:
    has_whitespace_issue = (orders_clean[col] != orders_clean[col].str.strip()).sum()
    print(f"{col}: 앞뒤 공백 있는 값 개수 = {has_whitespace_issue}")

=== Ship Mode (고유값 4개) ===
Ship Mode
Standard Class    6118
Second Class      1979
First Class       1548
Same Day           547
Name: count, dtype: int64

=== Segment (고유값 3개) ===
Segment
Consumer       5281
Corporate      3089
Home Office    1822
Name: count, dtype: int64

=== Country/Region (고유값 2개) ===
Country/Region
United States    9993
Canada            199
Name: count, dtype: int64

=== Region (고유값 4개) ===
Region
West       3253
East       2984
Central    2335
South      1620
Name: count, dtype: int64

=== Category (고유값 3개) ===
Category
Office Supplies    6128
Furniture          2199
Technology         1865
Name: count, dtype: int64

=== Sub-Category (고유값 17개) ===
Sub-Category
Binders        1548
Paper          1384
Furnishings    1008
Phones          903
Storage         856
Art             821
Accessories     775
Chairs          633
Appliances      474
Labels          368
Tables          326
Envelopes       256
Bookcases       232
Fasteners       229
Supplies        192
Machin

In [9]:
# 7. People / Returns 조인 무결성 확인

# People 테이블 확인
print("=== People ===")
print(people)
print()

# Orders의 Region이 People에 다 있는지
orders_regions = set(orders_clean["Region"].unique())
people_regions = set(people["Region"].unique())
print("Orders에만 있고 People에 없는 Region:", orders_regions - people_regions)
print("People에만 있고 Orders에 없는 Region:", people_regions - orders_regions)
print()

# Returns의 Order ID가 Orders에 다 존재하는지
orders_ids = set(orders_clean["Order ID"].unique())
returns_ids = set(returns["Order ID"].unique())
print("Returns 총 Order ID 개수:", len(returns_ids))
print("Returns에 있는데 Orders에 없는 Order ID 개수:", len(returns_ids - orders_ids))
print()

# Returns 자체에 중복 Order ID가 있는지 (같은 주문이 두 번 반품 기록됐는지)
print("Returns 내 Order ID 중복 개수:", returns["Order ID"].duplicated().sum())

=== People ===
    Regional Manager   Region
0    Sadie Pawthorne     West
1        Chuck Magee     East
2  Roxanne Rodriguez  Central
3        Fred Suzuki    South

Orders에만 있고 People에 없는 Region: set()
People에만 있고 Orders에 없는 Region: set()

Returns 총 Order ID 개수: 296
Returns에 있는데 Orders에 없는 Order ID 개수: 0

Returns 내 Order ID 중복 개수: 0


In [10]:
# State 단위 대시보드용 집계 테이블 생성
orders_clean["Order Year"] = orders_clean["Order Date"].dt.year

agg = (
    orders_clean
    .groupby(["State/Province", "Region", "Country/Region",
              "Order Year", "Category", "Segment"], as_index=False)
    .agg(
        총매출=("Sales", "sum"),
        총이익=("Profit", "sum"),
        주문건수=("Order ID", "nunique"),
        판매수량=("Quantity", "sum"),
        평균할인율=("Discount", "mean"),
    )
)

# 이익률 계산 (매출 0인 경우 대비)
agg["이익률"] = agg["총이익"] / agg["총매출"]

print(agg.shape)
print(agg.head(10))

# 대시보드용 CSV로 저장
agg.to_csv("superstore_state_dashboard_data.csv", index=False, encoding="utf-8-sig")
print("저장 완료: superstore_state_dashboard_data.csv")

(1243, 12)


  State/Province Region Country/Region  Order Year         Category  \
0        Alabama  South  United States        2023        Furniture   
1        Alabama  South  United States        2023        Furniture   
2        Alabama  South  United States        2023  Office Supplies   
3        Alabama  South  United States        2023  Office Supplies   
4        Alabama  South  United States        2023       Technology   
5        Alabama  South  United States        2023       Technology   
6        Alabama  South  United States        2024        Furniture   
7        Alabama  South  United States        2024        Furniture   
8        Alabama  South  United States        2024  Office Supplies   
9        Alabama  South  United States        2024  Office Supplies   

     Segment      총매출       총이익  주문건수  판매수량  평균할인율       이익률  
0   Consumer  1828.82  166.5650     2    16    0.0  0.091078  
1  Corporate  1761.80  387.1036     2    14    0.0  0.219721  
2   Consumer   174.70   80.77

In [11]:
print(agg["주문건수"].describe())
print()
print("주문건수 <= 2인 행 비율:", (agg["주문건수"] <= 2).mean())
print("주문건수 <= 5인 행 비율:", (agg["주문건수"] <= 5).mean())

count    1243.000000
mean        5.785197
std        10.050903
min         1.000000
25%         1.000000
50%         2.000000
75%         6.000000
max       133.000000
Name: 주문건수, dtype: float64

주문건수 <= 2인 행 비율: 0.5116653258246179
주문건수 <= 5인 행 비율: 0.7449718423169751


In [13]:
# 대시보드용 order-level 데이터 export (필요한 컬럼만)
dashboard_cols = [
    "Order ID", "Order Date", "State/Province", "Region", "Country/Region",
    "Category", "Segment", "Sales", "Quantity", "Discount", "Profit"
]

dashboard_data = orders_clean[dashboard_cols].copy()
dashboard_data["Order Year"] = dashboard_data["Order Date"].dt.year

print(dashboard_data.shape)
print(dashboard_data.dtypes)
print(dashboard_data.head())

import os

os.makedirs("data", exist_ok=True)
dashboard_data.to_csv("data/superstore_orders.csv", index=False, encoding="utf-8-sig")
print("저장 완료: data/superstore_orders.csv")

(10192, 12)
Order ID                     str
Order Date        datetime64[us]
State/Province               str
Region                       str
Country/Region               str
Category                     str
Segment                      str
Sales                    float64
Quantity                   int64
Discount                 float64
Profit                   float64
Order Year                 int32
dtype: object
         Order ID Order Date State/Province   Region Country/Region  \
0  US-2023-103800 2023-01-03          Texas  Central  United States   
1  US-2023-112326 2023-01-04       Illinois  Central  United States   
2  US-2023-112326 2023-01-04       Illinois  Central  United States   
3  US-2023-112326 2023-01-04       Illinois  Central  United States   
4  US-2023-141817 2023-01-05   Pennsylvania     East  United States   

          Category      Segment    Sales  Quantity  Discount   Profit  \
0  Office Supplies     Consumer   16.448         2       0.2   5.5512   
1  Of